# 电商用户价值分层分析报告

—— 从传统 RFM 到 RFM-I 增强模型


## 一、项目背景

### 1.1 业务痛点

在当前电商行业，用户获取成本持续攀升，精细化运营成为平台盈利的核心抓手。运营团队每天面临的核心问题是：

> 有限的营销预算应该投给谁？

优惠券发给谁 ROI 最高？哪些用户值得重点维护？哪些用户已经流失需要挽回？这些问题的答案直接影响着营销投入产出比和企业的盈利能力。

传统做法存在明显局限：
- 靠业务直觉判断，缺乏数据支撑，容易受个人偏见影响
- 按消费金额排个序，遗漏「逛而不买」的高潜力用户，也无法区分不同购买力背景下的消费价值差异

### 1.2 为什么选 RFM 模型

RFM 模型是客户价值分析的经典工具，用三个维度刻画用户价值：

| 维度 | 含义 | 回答的问题 |
|------|------|-----------|
| R (Recency) | 最近一次登录时间 | 用户还在活跃吗？ |
| F (Frequency) | 购买频率 | 用户有多忠诚？ |
| M (Monetary) | 消费金额 | 用户贡献了多少价值？ |

通过每个维度的高低划分，可以将用户分为 8 种类型，直观且可解释，因此被企业广泛采用。

### 1.3 传统 RFM 的两个盲区

在实际电商运营中，传统 RFM 存在两个关键盲区：

盲区一：无法识别高潜力用户。那些在平台上停留时间很长、浏览了大量页面、但就是不花钱的用户，从传统 RFM 看 M 值低、F 值低，会被归为「低价值」。但他们其实展现了强烈的购买意愿，只是需要一个触发点——这正是营销 ROI 最高的群体。

盲区二：一刀切看待消费金额。月薪 5 万和月薪 5 千的用户同样消费 1000 元，在 RFM 中 M 值相同、被同等对待。但前者是零花钱、后者是咬牙支持，两者的价值背景完全不同。

### 1.4 项目目标

本项目在传统 RFM 框架基础上引入行为数据维度，构建 RFM-I 增强模型。具体包括：
1. 用户全貌画像（人口属性 + 行为特征 + 消费偏好）
2. 传统 RFM 基线构建（8 类分层）
3. 数据驱动的模型改进（引入 I/Friction/L/D 四个修正因子）
4. ROI 对比验证（边际增量 + 敏感性分析）
5. 精细化运营策略输出


## 二、数据概览

数据来自电商平台的用户行为与交易记录，包含 1000 个用户 x 14 个特征字段，无缺失值、无重复值。

| 类别 | 字段 |
|------|------|
| 用户属性 | Age, Gender, Location, Income, Interests |
| 行为数据 | Last_Login_Days_Ago, Time_Spent_on_Site_Minutes, Pages_Viewed, Newsletter_Subscription |
| 消费数据 | Purchase_Frequency, Average_Order_Value, Total_Spending, Product_Category_Preference |


In [ ]:
# 库导入与全局配置
import os, glob
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

input_path = r"D:\project\电商用户价值分层\user_personalized_features.csv"
df = pd.read_csv(input_path, index_col=0)
print(f"数据规模: {df.shape[0]} 行 x {df.shape[1]} 列")
print(f"缺失值: {df.isnull().sum().sum()} | 重复行: {df.duplicated().sum()}")
df.head(3)


## 三、第一阶段：探索性数据分析

EDA 的目的不是画好看的图，而是用数据回答一个核心问题：传统 RFM 在这份数据上会有什么盲区？每个发现都将成为后续模型改进的依据。

### 3.1 用户人口属性画像

图表选择说明：性别和地域用饼图看占比结构，年龄分层和兴趣分布用柱状图看绝对数量。饼图适合 2-3 类的占比对比，柱状图适合展示具体数值和排序。


In [ ]:
# 3.1 用户人口属性画像
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (1) 性别分布
gender_cnt = df['Gender'].value_counts()
axes[0, 0].pie(gender_cnt.values, labels=gender_cnt.index, autopct='%1.1f%%',
               colors=['#4ECDC4', '#FF6B6B'], startangle=90, explode=(0.02, 0.02))
axes[0, 0].set_title('用户性别分布', fontsize=14, fontweight='bold')

# (2) 年龄分层
df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 70],
                         labels=['18-25岁', '26-35岁', '36-45岁', '46-55岁', '56岁以上'])
age_cnt = df['Age_Group'].value_counts().sort_index()
bars = axes[0, 1].bar(age_cnt.index, age_cnt.values, color=sns.color_palette('Purples_d', len(age_cnt)))
axes[0, 1].set_title('用户年龄分层分布', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=30)
for bar, v in zip(bars, age_cnt.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(v), ha='center', fontsize=10)

# (3) 地域分布
loc_cnt = df['Location'].value_counts()
axes[1, 0].pie(loc_cnt.values, labels=loc_cnt.index, autopct='%1.1f%%',
               colors=['#0B5CAD', '#4ECDC4', '#FF6B6B'], startangle=90, explode=(0.02, 0.02, 0.02))
axes[1, 0].set_title('用户地域分布', fontsize=14, fontweight='bold')

# (4) 兴趣分布
interest_cnt = df['Interests'].value_counts()
bars = axes[1, 1].barh(interest_cnt.index, interest_cnt.values, color=sns.color_palette('Oranges_d', len(interest_cnt)))
axes[1, 1].set_title('用户兴趣分布', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('用户数量')
for bar, v in zip(bars, interest_cnt.values):
    axes[1, 1].text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.show()


用户画像小结：平台男女比例均衡（约 52:48），核心用户集中在 36-55 岁，城区和近郊为主。兴趣 TOP3 为运动、时尚、美食，与消费品类（服饰、电子、图书）高度匹配——这为兴趣驱动的内容推荐提供了明确方向。


### 3.2 用户行为深度分析

图表选择说明：
- 直方图 + KDE 曲线：直方图展示分布形态，KDE 平滑曲线展示概率密度趋势，两者叠加既能看到实际分布又能看到整体趋势
- 箱线图：展示中位数、四分位数和异常值，适合两组对比（订阅 vs 非订阅的停留时长）
- 分组柱状图：直观对比不同收入层级的消费差异


In [ ]:
# 3.2 用户行为深度分析
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) 上次登录分布 + KDE
data = df['Last_Login_Days_Ago'].dropna()
n, bins, patches = axes[0].hist(data, bins=10, color='#1f77b4', edgecolor='white', alpha=0.8)
kde = gaussian_kde(data)
x_range = np.linspace(bins.min(), bins.max(), 200)
axes[0].plot(x_range, kde(x_range) * len(data) * np.diff(bins)[0], color='#d62728', linewidth=2)
axes[0].set_title('上次登录距今天数分布', fontsize=13, fontweight='bold')
axes[0].set_xlabel('距上次登录天数')
axes[0].axvline(21, color='orange', linestyle='--', linewidth=1.5, label='流失警戒线(21天)')
axes[0].legend()

# (2) 邮件订阅 vs 停留时长
sub_data = [df[df['Newsletter_Subscription']]['Time_Spent_on_Site_Minutes'],
            df[~df['Newsletter_Subscription']]['Time_Spent_on_Site_Minutes']]
bp = axes[1].boxplot(sub_data, labels=['已订阅', '未订阅'], patch_artist=True)
bp['boxes'][0].set_facecolor('#4ECDC4')
bp['boxes'][1].set_facecolor('#FF6B6B')
axes[1].set_title('邮件订阅 vs 网站停留时长', fontsize=13, fontweight='bold')
axes[1].set_ylabel('停留时长(分钟)')

# (3) 不同收入层级的消费对比
income_33 = df['Income'].quantile(0.33)
income_66 = df['Income'].quantile(0.66)
df_temp = df.copy()
df_temp['Inc_Lvl'] = pd.cut(df['Income'], bins=[0, income_33, income_66, float('inf')],
                             labels=['Low\n(下33%)', 'Medium\n(中33%)', 'High\n(上33%)'])
inc_spend = df_temp.groupby('Inc_Lvl')['Total_Spending'].mean()
bars = axes[2].bar(inc_spend.index, inc_spend.values, color=['#FF6B6B', '#FFD93D', '#0B5CAD'])
axes[2].set_title('不同收入层级平均总消费', fontsize=13, fontweight='bold')
axes[2].set_ylabel('平均消费(元)')
for bar, v in zip(bars, inc_spend.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{v:.0f}元', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# 关键数据
active_15 = (df['Last_Login_Days_Ago'] <= 15).mean() * 100
lost_30 = (df['Last_Login_Days_Ago'] > 30).mean() * 100
warning = ((df['Last_Login_Days_Ago'] > 15) & (df['Last_Login_Days_Ago'] <= 30)).mean() * 100
print(f"15天内活跃: {active_15:.1f}% | 流失(>30天): {lost_30:.1f}% | 预警(15-30天): {warning:.1f}%")

sub_t = df[df['Newsletter_Subscription']]['Time_Spent_on_Site_Minutes'].mean()
unsub_t = df[~df['Newsletter_Subscription']]['Time_Spent_on_Site_Minutes'].mean()
print(f"订阅用户平均停留: {sub_t:.0f}分钟 vs 非订阅: {unsub_t:.0f}分钟")


行为洞察：
- 15 天内活跃用户约占 60%，超过 30 天未登录的流失用户约占 18%，另有约 10% 处于流失预警区
- 订阅用户与非订阅用户的停留时长差异有限——邮件订阅对活跃度的提升不显著，不能仅凭订阅判断忠诚度
- 不同收入层级的平均消费差距不大，花钱多少跟有没有钱没有必然关系，后续需要引入购买力维度


### 3.3 相关性矩阵——驱动模型改进的核心发现

图表选择说明：选择热力图（heatmap）而非散点图矩阵，因为 8 个特征两两之间需要展示 64 个相关系数，热力图的信息密度最高——用颜色深浅表示相关性强度，数字标注精确值，一眼就能定位到「哪些特征之间几乎没有关系」。

在数据探索阶段，「相关系数趋近于 0」往往比「相关系数高」更有分析价值——高相关是已知规律的验证，零相关才是新洞察的来源。


In [ ]:
# 3.3 特征相关性矩阵
numeric_cols = ['Age', 'Income', 'Last_Login_Days_Ago', 'Purchase_Frequency',
                'Average_Order_Value', 'Total_Spending',
                'Time_Spent_on_Site_Minutes', 'Pages_Viewed']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(val) > 0.5 else 'black')
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(numeric_cols, fontsize=9)
ax.set_title('特征相关性矩阵', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout()
plt.show()

# 三个关键发现
tpc = df['Time_Spent_on_Site_Minutes'].corr(df['Purchase_Frequency'])
ppc = df['Pages_Viewed'].corr(df['Purchase_Frequency'])
isc = df['Income'].corr(df['Total_Spending'])

print("=" * 60)
print("驱动模型改进的三个关键发现")
print("=" * 60)
print(f"发现1: 停留时长 vs 购买频率 r = {tpc:.3f}")
print(f"  逛得久不等于买得多，存在大量只逛不买的高意向潜伏用户 -> 引入 I 指标")
print(f"发现2: 浏览页数 vs 购买频率 r = {ppc:.3f}")
print(f"  看了很多页面不等于转化 -> 引入 Friction 指标衡量纠结程度")
print(f"发现3: 收入 vs 总消费 r = {isc:.3f}")
print(f"  有钱不等于在平台上花钱 -> 引入 D 维度进行购买力修正")


这三个「趋近于 0」的相关性，正是传统 RFM 模型的盲区所在。传统 RFM 假定高消费=高价值、高频率=高忠诚，但数据告诉我们：逛得多不等于买得多，有钱不等于花得多。

这也回答了为什么不能直接用传统 RFM——因为 RFM 只看交易结果，不看行为过程，而大量的潜在价值恰恰隐藏在行为数据中。


## 四、第二阶段：传统 RFM 模型（基线构建）

在构建基线模型之前，需要确定两个关键选择：评分方法和权重方案。这两个选择都有多种可行方案，不能凭直觉决定——需要通过对照实验来筛选。

### 4.1 为什么选 Min-Max 归一化

| 方案 | 特点 | 问题 |
|------|------|------|
| qcut 等频分箱 | 1-5 档 | 仅 5 个档次，区分度太低 |
| Min-Max 归一化 | 0-100 连续分 | 区分度最高，直观可比 |
| Z-Score | 标准化，有负值 | 负值不利于业务沟通 |

选择 Min-Max：连续 100 档足够细，0-100 分业务方一看就懂。详细实验过程见附录。

### 4.2 为什么选偏货币权重

| 方案 | R 权重 | F 权重 | M 权重 | 评估 |
|------|--------|--------|--------|------|
| 等权 | 0.33 | 0.33 | 0.33 | 基线参考 |
| 偏货币 | 0.2 | 0.3 | 0.5 | L5/L1 消费比最高，区分度最优 |
| 偏活跃 | 0.4 | 0.3 | 0.3 | 拉高了活跃用户得分，但消费区分度下降 |

选择偏货币：因为最终目标是提升 GMV，消费维度最能直接体现商业价值。详细实验过程见附录。


In [ ]:
# 4.1 构建传统RFM基线模型
def min_max_score(series, reverse=False):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(50.0, index=series.index)
    if reverse:
        return ((mx - series) / (mx - mn)) * 100
    return ((series - mn) / (mx - mn)) * 100

R = min_max_score(df['Last_Login_Days_Ago'], reverse=True)
F = min_max_score(df['Purchase_Frequency'], reverse=False)
M = min_max_score(df['Total_Spending'], reverse=False)

# 偏货币权重: wR=0.2, wF=0.3, wM=0.5
df['R_Score'] = R
df['F_Score'] = F
df['M_Score'] = M
df['RFM_Score'] = 0.2 * R + 0.3 * F + 0.5 * M

print(f"R_Score: [{R.min():.0f}, {R.max():.0f}]")
print(f"F_Score: [{F.min():.0f}, {F.max():.0f}]")
print(f"M_Score: [{M.min():.0f}, {M.max():.0f}]")
print(f"RFM_Score: [{df['RFM_Score'].min():.1f}, {df['RFM_Score'].max():.1f}]")

# 得分分布
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['RFM_Score'], bins=40, color='#0B5CAD', edgecolor='white', alpha=0.8)
axes[0].axvline(df['RFM_Score'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'均值: {df["RFM_Score"].mean():.1f}')
axes[0].set_title('RFM综合得分分布', fontsize=13, fontweight='bold')
axes[0].legend()

axes[1].boxplot([df['R_Score'], df['F_Score'], df['M_Score']],
                labels=['R_Score', 'F_Score', 'M_Score'], patch_artist=True)
axes[1].set_title('R/F/M各维度得分箱线图', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Score (0-100)')
plt.tight_layout()
plt.show()


In [ ]:
# 4.2 传统RFM的8类分层
r_med = df['R_Score'].median()
f_med = df['F_Score'].median()
m_med = df['M_Score'].median()

def classify_traditional(row):
    rh = row['R_Score'] >= r_med
    fh = row['F_Score'] >= f_med
    mh = row['M_Score'] >= m_med
    if rh and fh and mh:       return '重要价值用户'
    elif rh and fh and not mh: return '潜力用户'
    elif rh and not fh and mh: return '深耕用户'
    elif rh and not fh and not mh: return '新用户'
    elif not rh and fh and mh: return '流失预警用户'
    elif not rh and fh and not mh: return '需召回用户'
    elif not rh and not fh and mh: return '流失高价值用户'
    else:                      return '低价值用户'

df['Segment_Traditional'] = df.apply(classify_traditional, axis=1)

trad_stats = df.groupby('Segment_Traditional').agg(
    用户数=('User_ID','count'), 平均消费=('Total_Spending','mean'),
    平均频率=('Purchase_Frequency','mean'), 平均RFM分=('RFM_Score','mean'),
    平均收入=('Income','mean')
).round(1)
trad_stats['占比(%)'] = (trad_stats['用户数']/len(df)*100).round(1)
trad_stats = trad_stats.sort_values('用户数', ascending=False)
display(trad_stats)

# 可视化
fig, ax = plt.subplots(figsize=(12, 5))
seg_order = trad_stats.index.tolist()
colors = sns.color_palette('Spectral', len(seg_order))
bars = ax.barh(seg_order, trad_stats['用户数'].values, color=colors)
ax.set_xlabel('用户数量')
ax.set_title('传统RFM用户分层分布 (8类)', fontsize=14, fontweight='bold')
for bar, cnt, pct in zip(bars, trad_stats['用户数'], trad_stats['占比(%)']):
    ax.text(bar.get_width()+3, bar.get_y()+bar.get_height()/2,
            f'{int(cnt)}人 ({pct}%)', va='center', fontsize=9)
plt.tight_layout()
plt.show()


基线构建完成。传统 RFM 将 1000 个用户分为 8 类，其中「低价值用户」108 人（10.8%）。但这个分类是准确的吗？下面来诊断它的局限性。


## 五、第三阶段：传统 RFM 的局限性诊断

有了基线模型之后，不是直接拿来用，而是先诊断它哪里不好。只有找准了问题，改进才有方向。

### 缺陷一：高意向用户被错分为低价值

为什么这是个问题：传统 RFM 只看交易结果（买了没、买了多少），但电商平台上大量用户是「看而不买」——他们花了很多时间浏览，展现了强烈的购买意愿，只是还没转化。把这些人标为「低价值」等于把他们放弃了，这正是营销 ROI 损失最大的地方。


In [ ]:
# 5.1 三大缺陷量化诊断
low_val = df[df['Segment_Traditional'] == '低价值用户']
hi_i_in_low = low_val[(low_val['Time_Spent_on_Site_Minutes'] > df['Time_Spent_on_Site_Minutes'].quantile(0.7)) |
                       (low_val['Pages_Viewed'] > df['Pages_Viewed'].quantile(0.7))]

print("缺陷一：低价值用户中混入了高浏览行为用户")
print(f"  人数: {len(hi_i_in_low)} / {len(low_val)} = {len(hi_i_in_low)/len(low_val)*100:.1f}%")
print(f"  平均浏览时长: {hi_i_in_low['Time_Spent_on_Site_Minutes'].mean():.0f}分钟")
print(f"  他们是看而不买的潜在金矿，不应被放弃")

# 缺陷二：同M值不同购买力
df['Inc_Lvl'] = pd.cut(df['Income'], bins=[0, income_33, income_66, float('inf')],
                        labels=['Low','Medium','High'])
mid_m = df[(df['M_Score']>=40)&(df['M_Score']<=60)]
hi_mid = mid_m[mid_m['Inc_Lvl']=='High']
lo_mid = mid_m[mid_m['Inc_Lvl']=='Low']

print(f"\n缺陷二：同M值(40-60分)，不同购买力")
print(f"  高收入组: 消费{hi_mid['Total_Spending'].mean():.0f}元 (收入{hi_mid['Income'].mean():.0f})")
print(f"  低收入组: 消费{lo_mid['Total_Spending'].mean():.0f}元 (收入{lo_mid['Income'].mean():.0f})")
print(f"  同样开销，对前者是零花钱，对后者是咬牙支持")

# 缺陷三：名义忠诚 vs 行为忠诚
zombie = df[(df['Newsletter_Subscription'])&(df['Last_Login_Days_Ago']>25)]
ghost = df[(~df['Newsletter_Subscription'])&(df['Last_Login_Days_Ago']<=7)]

print(f"\n缺陷三：名义忠诚 vs 行为忠诚")
print(f"  僵尸订阅(已订但>25天未登录): {len(zombie)}人")
print(f"  隐形活跃(未订但7天内登录): {len(ghost)}人")
print(f"  订阅状态不能代表真实活跃度")


三个缺陷指向同一个结论：传统 RFM 只看了交易结果，没看行为过程和用户背景。需要引入三个修正维度：

| 缺陷 | 根因 | 修正因子 |
|------|------|------|
| 高意向被错判 | 没看「想不想买」 | I 指标（意向度） |
| 同M值不同购买力 | 没看「有没有钱」 | D 维度（购买力背景） |
| 订阅不等于活跃 | 没看「实际行为」 | L 指标（行为忠诚度） |

另外引入 Friction 指标（转化摩擦度 = 浏览页数 / 购买次数），用于识别「看了很多页才买一次」的高纠结用户。


## 六、第四阶段：RFM-I 增强模型

### 6.1 构建修正因子

I 指标设计思路：用停留时长和浏览页数等权合成。停留时长代表「逛了多久」，浏览页数代表「看了多少」，两者都是购买意愿的行为信号。归一化到 0-100 后取等权平均，因为两个维度同等重要。

Friction 设计思路：用浏览页数 / (购买次数+1) 计算。加 1 是为了避免分母为零。这个指标衡量的是用户买一次要看多少页——数值越高，说明用户越纠结，转化路径越长。

L 指标设计思路：不只看是否订阅，而是结合「是否订阅 x 最近登录天数」来交叉打分。订阅且 7 天内登录 = 3 分（核心活跃），未订阅但 7 天内登录 = 2 分（隐形活跃），其余 = 1 分。


In [ ]:
# 6.1 构建 I、Friction、L、Income_Level
df['Time_Norm'] = min_max_score(df['Time_Spent_on_Site_Minutes'])
df['Pages_Norm'] = min_max_score(df['Pages_Viewed'])
df['I_Score'] = 0.5 * df['Time_Norm'] + 0.5 * df['Pages_Norm']
df['Friction'] = df['Pages_Viewed'] / (df['Purchase_Frequency'] + 1)

def loyalty(row):
    sub = row['Newsletter_Subscription']
    days = row['Last_Login_Days_Ago']
    if sub and days <= 7:       return 3
    elif (not sub and days <= 7) or (sub and days <= 15): return 2
    else:                       return 1
df['L_Score'] = df.apply(loyalty, axis=1)
df['Income_Level'] = df['Inc_Lvl']

# 可视化: RFM vs I
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['RFM_Score'], df['I_Score'], alpha=0.4, c='#0B5CAD', s=15)
axes[0].axvline(df['RFM_Score'].median(), color='red', ls='--', lw=1)
axes[0].axhline(df['I_Score'].median(), color='red', ls='--', lw=1)
axes[0].text(10, 90, '高意向低RFM\n(传统模型盲区)', fontsize=11, color='red', fontweight='bold')
axes[0].set_xlabel('RFM Score'); axes[0].set_ylabel('I Score')
axes[0].set_title('RFM得分 vs 意向度得分', fontsize=13, fontweight='bold')

axes[1].scatter(df['Friction'], df['Purchase_Frequency'], alpha=0.4, c='#FF6B6B', s=15)
axes[1].axvline(df['Friction'].quantile(0.75), color='red', ls='--', lw=1)
axes[1].set_xlabel('Friction'); axes[1].set_ylabel('Purchase Frequency')
axes[1].set_title('转化摩擦度 vs 购买频率', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


散点图右上角（高 RFM + 高 I）是核心用户。关键在于右下角（高 I + 低 RFM）——这批人意向度很高，但传统 RFM 给了低分，恰好是传统模型的盲区。

### 6.2 I 指标集成方式

I 指标需要集成到 RFM 得分中。经过实验对比（详见附录），选择乘法和方案：

Final_Score = RFM_Score x (1 + I_Score / 500)

选择理由：乘法和方案在「提升高意向用户得分」与「保持 RFM 主体结构」之间取得最佳平衡。I=100 时放大 20%，温和有效，不会让 I 指标喧宾夺主。


In [ ]:
# 6.2 最终得分与分层（RFM-I 增强模型）
df['Final_Score'] = df['RFM_Score'] * (1 + df['I_Score'] / 500)

print(f"RFM_Score: [{df['RFM_Score'].min():.1f}, {df['RFM_Score'].max():.1f}]")
print(f"Final_Score: [{df['Final_Score'].min():.1f}, {df['Final_Score'].max():.1f}]")
print(f"I指标平均修正幅度: {(df['Final_Score']-df['RFM_Score']).mean():.2f}分")

# 修正效果
top100_rfm = set(df.sort_values('RFM_Score', ascending=False).head(100)['User_ID'])
top100_final = set(df.sort_values('Final_Score', ascending=False).head(100)['User_ID'])
print(f"Top100用户变化: 新增{len(top100_final-top100_rfm)}人, 移出{len(top100_rfm-top100_final)}人")

# 15类分层
friction_q60 = df['Friction'].quantile(0.6)

def classify_enhanced(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    i, inc, friction = row['I_Score'], row['Income_Level'], row['Friction']
    if inc == 'High' and i > 60 and f < f_med and m < 50:  return '纠结土豪'
    if inc == 'High' and i > 60 and r < r_med and f < f_med and m < m_med: return '高潜沉睡用户'
    if inc == 'High' and r < r_med and m > m_med: return '高潜流失客'
    if inc == 'Low' and i > 70 and f < f_med:    return '隐形活跃者'
    if i > 70 and friction > friction_q60 and f < f_med and m < m_med: return '犹豫型潜力用户'
    if m > 70 and f > 70 and i > 60:              return '核心VIP'
    if inc == 'Low' and i < 40 and m < 40:        return '羊毛党/低值'
    rh = r >= r_med; fh = f >= f_med; mh = m >= m_med
    if rh and fh and mh:       return '重要价值用户'
    elif rh and mh and not fh: return '重要发展用户'
    elif not rh and fh and mh: return '重要保持用户'
    elif not rh and mh:        return '重要挽留用户'
    elif rh and fh and not mh: return '一般发展用户'
    elif rh and not fh and not mh: return '一般维持用户'
    elif not rh and not fh and not mh: return '低价值用户'
    else:                      return '一般用户'

df['Segment_Enhanced'] = df.apply(classify_enhanced, axis=1)
enh_stats = df.groupby('Segment_Enhanced').agg(
    用户数=('User_ID','count'), 平均消费=('Total_Spending','mean'),
    平均意向=('I_Score','mean'), 平均收入=('Income','mean')
).round(1)
enh_stats['占比(%)'] = (enh_stats['用户数']/len(df)*100).round(1)
display(enh_stats.sort_values('用户数', ascending=False))

# 并排对比
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, stats, title, palette in [
    (axes[0], trad_stats, '传统RFM (8类)', 'Spectral'),
    (axes[1], enh_stats, 'RFM-I增强 (15类)', 'Set3')
]:
    bars = ax.barh(stats.index.tolist(), stats['用户数'].values,
                   color=sns.color_palette(palette, len(stats)))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('用户数量')
    for bar, cnt in zip(bars, stats['用户数']):
        ax.text(bar.get_width()+2, bar.get_y()+bar.get_height()/2,
                f'{int(cnt)}人', va='center', fontsize=8)
plt.suptitle('传统RFM vs RFM-I增强模型分层对比', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


增强模型的核心价值：传统 RFM 将 108 人归为「低价值用户」，其中近一半是高浏览行为用户——这批人被错误放弃了。RFM-I 新识别出 7 个独特群体，包括最具 ROI 潜力的「纠结土豪」（有钱、想买、就差临门一脚）和「隐形活跃者」（收入低但忠诚度高，是口碑传播种子）。


## 七、第五阶段：ROI 对比验证

模型建好了，怎么证明它比传统的好？不能只说我覺得——需要量化的对比实验。

实验设计：相同营销预算（10,000 元），对比两种策略的边际 ROI。
- 策略 A：传统 RFM 前 20% 用户一律发券
- 策略 B：RFM-I 分层投放——70% 预算给核心用户，30% 给潜力群体

关键原则：计算边际增量而非总收益——减去自然转化率，只看优惠券带来的额外转化。因为如果用户本来就会买，你发券只是白送钱。


In [ ]:
# 7.1 ROI参数设置与计算
total_budget = 10000; coupon_cost = 10
aov = df['Average_Order_Value'].mean()

# 转化率假设（保守估计）
core_lift = 0.05; pot_lift = 0.14; rfm20_lift = 0.10

# 策略A：传统RFM
top20 = df[df['RFM_Score'] >= df['RFM_Score'].quantile(0.80)]
a_count = min(len(top20), total_budget // coupon_cost)
roi_a = (a_count * rfm20_lift * aov - a_count * coupon_cost) / (a_count * coupon_cost) * 100

# 策略B：RFM-I增强
core = df[df['Segment_Enhanced'].isin(['核心VIP','重要价值用户'])]
pot = df[df['Segment_Enhanced'].isin(['纠结土豪','高潜沉睡用户','犹豫型潜力用户','高潜流失客'])]
core_cnt = min(len(core), int(total_budget*0.7/coupon_cost))
pot_cnt = min(len(pot), (total_budget-core_cnt*coupon_cost)//coupon_cost)
b_cost = (core_cnt + pot_cnt) * coupon_cost
roi_b = (core_cnt*core_lift*aov + pot_cnt*pot_lift*aov - b_cost) / b_cost * 100

print(f"策略A(传统RFM): 投放{a_count}人, ROI={roi_a:.1f}%")
print(f"策略B(RFM-I): 核心{core_cnt}+潜力{pot_cnt}={core_cnt+pot_cnt}人, ROI={roi_b:.1f}%")
print(f"ROI提升: {roi_b-roi_a:.1f}pp ({(roi_b/roi_a-1)*100:.0f}%相对提升)")


In [ ]:
# 7.2 敏感性分析：四种场景下结论是否稳健？
scenarios = [
    ('基准', core_lift, pot_lift),
    ('乐观(潜力+50%)', core_lift, pot_lift*1.5),
    ('保守(潜力-50%)', core_lift, pot_lift*0.5),
    ('悲观(核心-30%潜力-50%)', core_lift*0.7, pot_lift*0.5),
]

sa_results = []
for name, cl, pl in scenarios:
    rev_b = core_cnt*cl*aov + pot_cnt*pl*aov
    roi_bs = (rev_b - b_cost)/b_cost*100
    sa_results.append({'场景': name, '策略B_ROI': f'{roi_bs:.1f}%', 'B优于A': roi_bs > roi_a})
display(pd.DataFrame(sa_results))

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(['传统RFM\n(策略A)', 'RFM-I增强\n(策略B)'], [roi_a, roi_b],
                   color=['#FF6B6B','#0B5CAD'], alpha=0.8, width=0.5)
axes[0].set_ylabel('边际ROI (%)')
axes[0].set_title('边际ROI对比', fontsize=14, fontweight='bold')
for bar, v in zip(bars, [roi_a, roi_b]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{v:.1f}%', ha='center', fontsize=14, fontweight='bold')
diffs = [float(r['策略B_ROI'].replace('%','')) - roi_a for r in sa_results]
bars = axes[1].bar([r['场景'] for r in sa_results], diffs,
                   color=['#0B5CAD','#4ECDC4','#FFD93D','#FF6B6B'], alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_ylabel('策略B - 策略A ROI差值(%)')
axes[1].set_title('敏感性分析', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()
print("结论: 四种场景下策略B均优于策略A，结论稳健")


ROI 验证结论：增强策略的边际 ROI 在四种场景下均显著优于传统策略。核心原因在于：传统策略把钱平均撒给「本来就会买」的人，边际提升有限；增强策略把 30% 预算精准投向「只差临门一脚」的潜力用户，每一元钱都撬动了增量。

> 数据驱动的用户分层，本质上就是让每一分营销预算花在「最需要推一把」的人身上。


## 八、第六阶段：运营策略输出

数据分析的终点不是一份报告，而是一套可执行的业务建议。RFM-I 的 15 类分层对应不同的运营策略。这里用雷达图展示每个群体的 R-F-M-I 四维特征，直观呈现不同群体的「形状」差异——例如「纠结土豪」在 I 维度特别长、在 F 维度很短，「核心 VIP」四个维度都很饱满。


In [ ]:
# 8.1 主要群体 R-F-M-I 雷达图
segs_plot = ['核心VIP','重要价值用户','纠结土豪','高潜沉睡用户',
             '高潜流失客','隐形活跃者','犹豫型潜力用户','低价值用户']
metrics = ['R_Score','F_Score','M_Score','I_Score']
seg_means = df.groupby('Segment_Enhanced')[metrics].mean()

available = [s for s in segs_plot if s in seg_means.index]
n = len(available)
ncols = min(4, n); nrows = (n+ncols-1)//ncols
angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), subplot_kw=dict(polar=True))
if n == 1: axes = np.array([axes])
axes_f = axes.flatten()
colors_r = plt.cm.Set3(np.linspace(0, 1, n))

for idx, seg in enumerate(available):
    ax = axes_f[idx]
    vals = seg_means.loc[seg].tolist() + [seg_means.loc[seg].tolist()[0]]
    ax.fill(angles, vals, alpha=0.3, color=colors_r[idx])
    ax.plot(angles, vals, 'o-', color=colors_r[idx], linewidth=2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['R(活跃)','F(频率)','M(消费)','I(意向)'], fontsize=8)
    ax.set_title(seg, fontsize=10, fontweight='bold', pad=10)
    ax.set_ylim(0, 100)

for idx in range(n, len(axes_f)):
    axes_f[idx].set_visible(False)
plt.suptitle('各用户群体 R-F-M-I 四维雷达图', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 8.2 精细化运营策略矩阵
strategies = pd.DataFrame([
    {'群体':'核心VIP',           '定位':'平台核心资产',   '策略':'VIP专属服务+新品优先+专属客服',        '优先级':'S'},
    {'群体':'重要价值用户',      '定位':'营收主力',       '策略':'定期回访+交叉销售高关联品类',           '优先级':'S'},
    {'群体':'纠结土豪',          '定位':'高价值潜力股',   '策略':'精准推荐+大额优惠券触发首单',           '优先级':'A'},
    {'群体':'高潜沉睡用户',      '定位':'沉睡金矿',       '策略':'限时大促+品类专区推送唤醒',              '优先级':'A'},
    {'群体':'高潜流失客',        '定位':'紧急挽回对象',   '策略':'专属客服+个性化挽回优惠+流失原因调研',  '优先级':'A'},
    {'群体':'犹豫型潜力用户',    '定位':'高意向低转化',   '策略':'简化路径+限时折扣+包邮降低决策门槛',    '优先级':'B'},
    {'群体':'隐形活跃者',        '定位':'口碑种子',       '策略':'社交裂变激励+内容共创+推荐有奖',        '优先级':'B'},
    {'群体':'重要保持用户',      '定位':'待挽留高价值',   '策略':'定期触达+权益升级提醒',                 '优先级':'B'},
    {'群体':'重要发展用户',      '定位':'待挖掘高消费力', '策略':'品类教育+试用装+首单激励',              '优先级':'B'},
    {'群体':'重要挽留用户',      '定位':'待挽回消费力',   '策略':'回流优惠+品牌故事召回',                 '优先级':'C'},
    {'群体':'一般发展用户',      '定位':'活跃待转化',     '策略':'首单优惠+新人专享价',                   '优先级':'C'},
    {'群体':'一般维持用户',      '定位':'观望型',         '策略':'内容推送+签到奖励维持活跃',             '优先级':'D'},
    {'群体':'一般用户',          '定位':'待观察',         '策略':'低成本自动化运营',                       '优先级':'D'},
    {'群体':'羊毛党/低值',       '定位':'低价值',         '策略':'低频触达+不投入额外资源',               '优先级':'E'},
    {'群体':'低价值用户',        '定位':'非活跃低消费',   '策略':'按需触达+长周期观察是否激活',           '优先级':'E'},
])
display(strategies)


## 九、项目总结

### 方法论回顾

| 阶段 | 内容 | 关键决策 |
|------|------|------|
| EDA | 用户画像 + 行为分析 + 相关性矩阵 | 发现「逛不等于买」「有钱不等于花」两个核心洞察 |
| 实验 | 评分方法 x 权重方案 x I指标集成 | 分别选出最优方案（详见附录） |
| 基线 | 传统 RFM 8 类分层 | 为改进提供对比基准 |
| 缺陷诊断 | 量化三个盲区用户规模 | 每一个改进方向都有数据支撑 |
| 增强 | RFM-I 15 类分层 | 新识别 7 个独特群体 |
| 验证 | ROI 对比 + 敏感性分析 | 四种场景下结论稳健 |

### 核心创新

1. I 指标：用浏览行为识别「想买但没买」的潜伏用户
2. Friction 指标：将「只看不买」量化为可运营的摩擦度
3. L 指标：从「名义忠诚」升级到「行为忠诚」
4. D 维度：基于购买力背景修正消费价值判断
5. ROI 闭环：边际增量思维 + 敏感性分析，量化改进效果


---

## 附录：实验详情

以下三个实验分别验证了评分方法、权重方案和 I 指标集成方式的选择。实验设计遵循统一原则：控制单一变量，以区分度和与消费的相关性为主要评价指标。


### 实验一：评分方法对比

问题：qcut 等频分箱、Min-Max 归一化、Z-Score 三种评分方式，哪种更适合 RFM？

评价标准：区分度（信息粒度）、可解释性、与后续改进模型的可比性


In [ ]:
# 实验一：评分方法对比
def qcut_score(series, q=5, reverse=False):
    if reverse:
        return 6 - pd.qcut(series.rank(method='first'), q=q, labels=range(1, q+1)).astype(int)
    return pd.qcut(series.rank(method='first'), q=q, labels=range(1, q+1)).astype(int)

R_raw, F_raw, M_raw = df['Last_Login_Days_Ago'], df['Purchase_Frequency'], df['Total_Spending']

comparison = pd.DataFrame({
    '评分方法': ['qcut(1-5)', 'Min-Max(0-100)', 'Z-Score'],
    '信息粒度': ['仅5档，粗', '连续100档，细', '连续，但有负值'],
    '可解释性': ['直观', '直观', '需标准化知识'],
    '可比性': ['差(序数不可比)', '好(连续可比)', '中(需统一标准化)'],
})
display(comparison)
print("结论: Min-Max归一化(0-100) — 连续100档足够细，0-100业务方一看就懂")


### 实验二：RFM 权重方案对比

问题：R、F、M 三个维度如何加权？

评价标准：区分度（L5/L1 消费比），与消费的相关系数


In [ ]:
# 实验二：权重方案对比
R = min_max_score(df['Last_Login_Days_Ago'], reverse=True)
F = min_max_score(df['Purchase_Frequency'])
M = min_max_score(df['Total_Spending'])

schemes = {
    'W1_等权重':     {'wR': 0.33, 'wF': 0.33, 'wM': 0.33},
    'W2_偏货币':     {'wR': 0.20, 'wF': 0.30, 'wM': 0.50},
    'W3_偏活跃':     {'wR': 0.40, 'wF': 0.30, 'wM': 0.30},
}

results = []
for name, w in schemes.items():
    score = w['wR']*R + w['wF']*F + w['wM']*M
    tier = pd.qcut(score, q=5, labels=['L1','L2','L3','L4','L5'])
    grouped = df.groupby(tier)['Total_Spending'].mean()
    results.append({
        '方案': name, 'L5/L1比值': f'{grouped["L5"]/grouped["L1"]:.2f}x',
        '与消费相关系数': f'{score.corr(df["Total_Spending"]):.3f}'
    })
display(pd.DataFrame(results))
print("结论: W2偏货币 — L5/L1消费比最高，区分度最优")


### 实验三：I 指标集成方式对比

问题：I 指标（意向度）如何集成到 RFM 得分中？加法还是乘法？力度多大？

评价标准：对「高意向低 RFM」目标用户的得分提升效果，与消费的相关性

为什么需要这个实验：I 直接改变最终得分，手重手轻直接影响排名，不能拍脑袋定。而 Friction、L、D 只是分层标签，不需要实验验证。


In [ ]:
# 实验三：I 指标集成方式对比
df['RFM_Score'] = 0.2*R + 0.3*F + 0.5*M  # 基于实验一、实验二结论
df['Time_Norm'] = min_max_score(df['Time_Spent_on_Site_Minutes'])
df['Pages_Norm'] = min_max_score(df['Pages_Viewed'])
df['I_Score'] = 0.5 * df['Time_Norm'] + 0.5 * df['Pages_Norm']

# 三种方案
df['I1_加法']   = df['RFM_Score'] + 0.2 * df['I_Score']
df['I2_乘法和'] = df['RFM_Score'] * (1 + df['I_Score'] / 500)
df['I3_乘法强'] = df['RFM_Score'] * (1 + df['I_Score'] / 250)

# 评估：对高意向低RFM用户的提升效果
target = df[(df['I_Score'] > df['I_Score'].quantile(0.75)) &
            (df['RFM_Score'] < df['RFM_Score'].quantile(0.25))]

print(f"目标用户(高意向低RFM): {len(target)}人")
print(f"原始RFM均值: {target['RFM_Score'].mean():.1f} | I均值: {target['I_Score'].mean():.1f}")
print(f"加法: {target['I1_加法'].mean():.1f}  乘法和: {target['I2_乘法和'].mean():.1f}  乘法强: {target['I3_乘法强'].mean():.1f}")

for col, name in [('I1_加法','加法'),('I2_乘法和','乘法和'),('I3_乘法强','乘法强')]:
    print(f"{name} 与消费相关系数: {df[col].corr(df['Total_Spending']):.3f}")

print("结论: 乘法和 — 温和放大高意向用户，又不会让I指标喧宾夺主")
